# 04 · Procedencia del dato, faltantes, duplicados y mapa de las estaciones

**Laboratorio en Python, no sistema.** Go sigue siendo la fuente de verdad del silver y del gold; este
notebook no limpia nada para el modelo. Responde tres preguntas del coordinador sobre el bronze de OEFA:

1. **¿Dónde están las estaciones?** Ficha por estación (`reports/estaciones_ficha.csv`), mapa interactivo
   (`reports/mapa_estaciones.html`, Leaflet sobre OpenStreetMap) y mapa estático del informe (`mapa.png`).
2. **¿Por qué faltan valores y por qué hay duplicados?** Cada fila larga se etiqueta con la regla de la
   cascada de Go que la descarta (los conteos se cuadran con `reports/limpieza.json`), y se pregunta
   *por qué*: hora `NA`, días enteros en cero, filas sin ningún valor, gases en dos unidades, promedios de
   24 h y móviles, estaciones sin horarios. Calendario de disponibilidad (`calendario.png`) y
   `faltantes.png`; tablas `reports/faltantes_*.csv`.
3. **¿Cómo se recolectó el dato?** Lo que dicen las columnas de contexto, el diccionario y los metadatos de
   OEFA. El texto largo, con las fuentes web citadas, está en `docs/eda-procedencia.md`.

Convención de este notebook: se separa **lo que el dato muestra** (una cifra calculada aquí), **lo que dice
OEFA** (diccionario, metadatos, informes) y **lo que es inferencia** (se marca como tal).

| Entra | Sale |
|---|---|
| `data/bronze/*.csv` y su diccionario (`1c_*.xlsx`) y metadatos (`1b_*.docx`) | `reports/estaciones_ficha.csv`, `reports/mapa_estaciones.html` |
| `configs/limpieza.json`, `reports/limpieza.json` (Go) | `reports/faltantes_por_estacion.csv`, `faltantes_tramos_en_cero.csv`, `faltantes_sin_valor.csv`, `faltantes_promedio_24h.csv` |
| `data/silver/oefa_ancho.csv`, `data/gold/oefa_features.csv`, `data/gold/asignaciones.csv`, `reports/regimenes.json` (opcionales) | figuras del informe `mapa.png`, `faltantes.png`, `calendario.png` |

## 0. Entorno

En Colab basta con *Entorno de ejecución → Ejecutar todo*. Esta celda monta Drive (la primera vez en
cada sesión pide autorizarlo), deja el código en la rama con que se abrió el notebook, instala lo que
falte, imprime commit, raíz de los datos, CPU y RAM, y verifica que los insumos estén en
`MyDrive/aire-distribuido/`: si falta algo o está desfasado, dice qué subir y con qué comando de
`rclone`. Ver «Iterar en Colab» en `notebooks/README.md`.

In [ ]:
# ── 0. Entorno: la misma celda en los cuatro notebooks ─────────────────────────────────────────
# En Colab: monta Drive, deja el repo en /content/aire-distribuido en la rama con que se abrió el
# notebook (la de la URL; si no se puede leer, RAMA) con git fetch + reset --hard, instala lo que
# falte y dice qué commit, qué datos y qué máquina se usan. En local no toca git ni Drive: usa el
# repo donde está el notebook y la raíz de AIRE_RAIZ (o el repo).
RAMA = "feature/spec-pc3"   # rama del código si no se puede leer de la URL de Colab
ACTUALIZAR_CODIGO = True    # Colab: traer lo último de la rama en cada corrida (False: dejar el código como está)
RAIZ_DRIVE = "/content/drive/MyDrive/aire-distribuido"  # capas medallón en Drive
NOTEBOOK = "04"              # qué insumos se verifican al final de la celda

import importlib, importlib.util, json, os, re, subprocess, sys, time, urllib.parse, urllib.request
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
RAIZ_LOCAL = os.environ.get("AIRE_RAIZ")  # otra raíz de las capas (p. ej. el sintético); en Colab, sin Drive
REPO_GIT = "https://github.com/R0SEWT/aire-distribuido.git"
PAQUETES = {"pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib", "scikit-learn": "sklearn",
            "scipy": "scipy", "joblib": "joblib"}  # Colab los trae: solo se instala lo que falte
if NOTEBOOK == "04":  # el mapa interactivo; contextily (teselas del mapa estático) es opcional
    PAQUETES["folium"] = "folium"
OPCIONALES = {"contextily": "contextily"} if NOTEBOOK == "04" else {}

if EN_COLAB:  # Colab corre en UTC: las fechas de reportes y figuras, en hora de Lima
    os.environ["TZ"] = "America/Lima"
    time.tzset()
if EN_COLAB and not RAIZ_LOCAL:
    from google.colab import drive
    # Cada entorno nuevo pide autorizar Drive; con «Ejecutar todo» se espera hasta 15 min (no los 2 por defecto).
    drive.mount("/content/drive", timeout_ms=15 * 60 * 1000)


def rama_de_colab():
    """La rama de la URL del notebook (colab…/github/R0SEWT/aire-distribuido/blob/<rama>/notebooks/…)."""
    for host in dict.fromkeys([os.environ.get("COLAB_JUPYTER_IP"), "172.28.0.12", "172.28.0.2", "127.0.0.1"]):
        if not host:
            continue
        try:
            puerto = os.environ.get("KMP_TARGET_PORT", "9000")
            with urllib.request.urlopen(f"http://{host}:{puerto}/api/sessions", timeout=3) as r:
                texto = urllib.parse.unquote(urllib.parse.unquote(r.read().decode()))
        except Exception:
            continue
        m = re.search(r"aire-distribuido/blob/(.+?)/notebooks/", texto)
        return m.group(1) if m else None
    return None


def git(*args, repo=None):
    r = subprocess.run(["git", *(["-C", str(repo)] if repo else []), *args], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"git {' '.join(args)}: {r.stderr.strip()}")
    return r.stdout.strip()


def traer_repo(rama, destino=Path("/content/aire-distribuido")):
    """Clona la rama (la primera vez) o la pone al día con fetch + reset --hard (las siguientes)."""
    if not (destino / ".git").exists():
        subprocess.run(["rm", "-rf", str(destino)])
        git("clone", "-q", "--depth", "1", "-b", rama, REPO_GIT, str(destino))
    elif ACTUALIZAR_CODIGO:
        git("fetch", "-q", "--depth", "1", "origin", f"+refs/heads/{rama}:refs/remotes/origin/{rama}", repo=destino)
        git("checkout", "-q", "-f", "-B", rama, f"origin/{rama}", repo=destino)
        git("reset", "-q", "--hard", f"origin/{rama}", repo=destino)
    return destino


if EN_COLAB:
    RAMA_USADA, ORIGEN_RAMA = rama_de_colab(), "de la URL de Colab"
    if not RAMA_USADA:
        RAMA_USADA, ORIGEN_RAMA = RAMA, "de RAMA: no se pudo leer la URL"
    for rama in dict.fromkeys([RAMA_USADA, "develop"]):
        try:
            REPO = traer_repo(rama)
            break
        except RuntimeError as err:
            print(f"AVISO: no pude traer la rama {rama!r} ({err}); pruebo la siguiente.")
    else:
        raise RuntimeError(f"No pude clonar {REPO_GIT}: ¿hay internet en el entorno de ejecución?")
    if rama != RAMA_USADA:
        RAMA_USADA, ORIGEN_RAMA = rama, f"{RAMA_USADA!r} no existe en GitHub"
    if git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO) != RAMA_USADA:  # clon viejo sin actualizar
        RAMA_USADA, ORIGEN_RAMA = git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO), "clon anterior: ACTUALIZAR_CODIGO = False"
else:
    REPO = next((c for c in (Path.cwd(), Path.cwd().parent) if (c / "notebooks" / "aire_lab.py").exists()), None)
    if REPO is None:
        raise FileNotFoundError("No encuentro notebooks/aire_lab.py: abre el notebook desde notebooks/ o desde la raíz del repo.")
    RAMA_USADA, ORIGEN_RAMA = None, ""

faltan = [pip for pip, mod in PAQUETES.items() if importlib.util.find_spec(mod) is None]
if faltan:
    print("Instalando:", faltan)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
for pip, mod in OPCIONALES.items():  # si no se instala, el notebook sigue (sin teselas en el mapa estático)
    if importlib.util.find_spec(mod) is None:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip], check=False)
        print(f"{pip}: {'instalado' if r.returncode == 0 else 'no se pudo instalar (opcional)'}")

if str(REPO / "notebooks") not in sys.path:
    sys.path.insert(0, str(REPO / "notebooks"))
RAIZ = Path(RAIZ_LOCAL) if RAIZ_LOCAL else (Path(RAIZ_DRIVE) if EN_COLAB else REPO)
DATA, REPORTS = RAIZ / "data", RAIZ / "reports"
CONFIGS = [RAIZ / "configs", REPO / "configs"]  # la de la raíz (Drive), si existe, va primero
# Figuras del informe: en local, docs/informe/generado/figuras/; en Colab, reports/figuras/ de Drive.
FIGURAS = Path(os.environ["AIRE_FIGURAS"]) if os.environ.get("AIRE_FIGURAS") else (
    RAIZ / "reports" / "figuras" if EN_COLAB else REPO / "docs" / "informe" / "generado" / "figuras")
REPORTS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

import aire_lab as al
importlib.reload(al)
CMAPS = al.estilo_figuras()
SINTETICO = al.es_sintetico(RAIZ)  # se vuelve a mirar al leer los datos (estaciones SINT-/SIN-)
ENTORNO = al.describir_entorno(REPO, RAIZ, FIGURAS, EN_COLAB, RAMA_USADA, ORIGEN_RAMA)
INSUMOS = al.verificar_insumos(NOTEBOOK, RAIZ, CONFIGS, estricto=EN_COLAB)

In [ ]:
import html, json, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from IPython.display import display, Markdown

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 90)

NOMBRE_NB = "04_procedencia_y_mapa"  # para origen.json; NOTEBOOK = "04" es el de la celda de entorno


def guardar(fig, nombre, fuente, dpi=150):
    """Figura del informe: se guarda en FIGURAS y se anota su origen (origen.json).
    dpi=300 para las figuras dibujadas ya al ancho final del informe (figsize en pulgadas = cm/2,54)."""
    al.marca_sintetico(fig, SINTETICO)
    fig.savefig(FIGURAS / nombre, dpi=dpi)
    al.registrar_figura(FIGURAS, nombre, SINTETICO, fuente, NOMBRE_NB)
    plt.show()


def md(texto):
    display(Markdown(texto))


def tabla(df, decimales=1, indice=False):
    md(al.tabla_md(df, decimales=decimales, indice=indice))

## 1. El bronze completo, con todas sus columnas

Los notebooks 01 y 03 leen solo las columnas que usa Go. Aquí se leen **las 36**, porque las de contexto
(`NOMBRE_EVALUACION`, `TIPO_MUESTRA`, `TIPO_ANALISIS`, `PERIODO`, `DESCRIPCION_UBICACION`, `FECHA_CORTE`…)
son las que cuentan cómo se recolectó el dato. Se leen como texto por trozos y se guardan como categorías
(unos 150 MB en memoria). pandas une los 100 registros con un salto de línea dentro de un campo; Go los
rechaza (ver el 01).

In [ ]:
BRONZE = os.environ.get("AIRE_BRONZE")  # None: el CSV más grande de data/bronze/
dir_bronze = DATA / "bronze"
if BRONZE is None:
    candidatos = sorted(dir_bronze.glob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
    if not candidatos:
        raise FileNotFoundError(f"No hay ningún CSV en {dir_bronze}: sube el bronze de OEFA a Drive.")
    BRONZE = candidatos[0]
BRONZE = Path(BRONZE)
SINTETICO = SINTETICO or al.es_sintetico(bronze=BRONZE)


def leer_completo(ruta, trozo=400_000) -> pd.DataFrame:
    """Todas las columnas como texto -> categoría, por trozos (el orden de las filas es el del archivo)."""
    from pandas.api.types import union_categoricals
    partes, columnas = None, None
    for t in pd.read_csv(ruta, sep=";", dtype=str, keep_default_na=False, na_values=[], chunksize=trozo,
                         encoding="utf-8", encoding_errors="replace"):
        if partes is None:
            columnas = list(t.columns)
            partes = {c: [] for c in columnas}
        for c in columnas:
            partes[c].append(t[c].astype("category"))
    return pd.DataFrame({c: pd.Series(union_categoricals(partes[c])) for c in columnas})


t0 = time.time()
B = leer_completo(BRONZE)
print(f"{BRONZE.name}: {len(B):,} filas × {B.shape[1]} columnas en {time.time() - t0:.0f} s "
      f"({B.memory_usage(deep=True).sum() / 1e6:.0f} MB en memoria)")
if SINTETICO:
    print("AVISO: datos sintéticos: las cifras prueban el notebook, no dicen nada del aire del Perú.")

### 1.1 Lo que dice OEFA del dataset: diccionario y metadatos

El diccionario (`1c_*.xlsx`) y la ficha de metadatos (`1b_*.docx`) vienen junto al CSV. Se leen sin
dependencias extra (son ZIP con XML). Es **lo que OEFA dice**, no lo que el dato muestra.

In [ ]:
def leer_xlsx(ruta) -> list[list[str]]:
    """Filas de la primera hoja de un .xlsx (texto), sin openpyxl."""
    z = zipfile.ZipFile(ruta)
    cadenas = []
    if "xl/sharedStrings.xml" in z.namelist():
        x = z.read("xl/sharedStrings.xml").decode("utf-8")
        cadenas = [html.unescape(re.sub(r"<[^>]+>", "", s)) for s in re.findall(r"<si>(.*?)</si>", x, flags=re.S)]
    hoja = sorted(n for n in z.namelist() if n.startswith("xl/worksheets/sheet"))[0]
    x = z.read(hoja).decode("utf-8")
    filas = []
    for fila in re.findall(r"<row[^>]*>(.*?)</row>", x, flags=re.S):
        celdas = []
        for ref, attrs, interior in re.findall(r'<c r="([A-Z]+)\d+"([^>]*?)(?:/>|>(.*?)</c>)', fila, flags=re.S):
            v = re.search(r"<v>(.*?)</v>", interior or "")
            if v:
                v = v.group(1)
                celdas.append(cadenas[int(v)] if 't="s"' in attrs else v)
        if celdas:
            filas.append(celdas)
    return filas


def leer_docx(ruta) -> str:
    x = zipfile.ZipFile(ruta).read("word/document.xml").decode("utf-8")
    x = re.sub(r"</w:p>", "\n", x)
    x = re.sub(r"</w:tc>", " | ", x)
    return html.unescape(re.sub(r"<[^>]+>", "", x))


dic = next(iter(sorted(dir_bronze.glob("1c_*.xlsx"))), None)
meta = next(iter(sorted(dir_bronze.glob("1b_*.docx"))), None)
DICCIONARIO = {}
if dic:
    for f in leer_xlsx(dic):
        if len(f) >= 2 and re.fullmatch(r"[A-Z0-9_.]+", f[0]):
            DICCIONARIO[f[0]] = {"descripcion": f[1], "tipo": f[2] if len(f) > 2 else "",
                                 "adicional": f[-1] if len(f) > 5 else ""}
    contexto = ["NOMBRE_EVALUACION", "COMPONENTE_AMBIENTAL", "PROCEDENCIA_MUESTRA", "NOMBRE_PUNTO",
                "DESCRIPCION_UBICACION", "TIPO_MUESTRA", "TIPO_ANALISIS", "PERIODO", "FECHA_INICIO",
                "HORA_INICIO", "FECHA_FIN", "HORA_FIN", "ZONA", "FECHA_CORTE"]
    tabla(pd.DataFrame([{"columna": c, **DICCIONARIO.get(c, {})} for c in contexto]))
else:
    print("No está el diccionario (1c_*.xlsx) en data/bronze/.")
if meta:
    texto = leer_docx(meta)
    for clave in ["Descripción", "Entidad", "Fuente", "Fecha de creación", "Frecuencia de actualización",
                  "Última actualización", "Cobertura"]:
        m = re.search(r"\|\s*" + re.escape(clave) + r"\s*\n\s*\|\s*(.+?)\s*(?:\n\s*\||$)", texto, flags=re.S)
        if m:
            print(f"{clave}: {' '.join(m.group(1).split())[:900]}")
else:
    print("No está la ficha de metadatos (1b_*.docx) en data/bronze/.")

### 1.2 Las columnas de contexto, tal como vienen

Lo que el dato muestra, columna por columna: cuántos valores distintos y cuáles.

In [ ]:
CONTEXTO = ["COMPONENTE_AMBIENTAL", "PROCEDENCIA_MUESTRA", "TIPO_MUESTRA", "TIPO_ANALISIS", "PERIODO",
            "UNIDAD_MEDIDA", "FECHA_FIN", "HORA_FIN", "FECHA_CORTE", "DATUM", "ZONA"]
filas = []
for c in CONTEXTO:
    vc = B[c].value_counts()
    filas.append({"columna": c, "distintos": len(vc),
                  "valores (filas)": "; ".join(f"«{k}» {al.miles(v)}" for k, v in vc.head(9).items())})
tabla(pd.DataFrame(filas))
fechas_txt = B["FECHA_INICIO"].astype(str)
print(f"{B['NOMBRE_EVALUACION'].nunique()} evaluaciones, {B['NOMBRE_PUNTO'].nunique()} estaciones "
      f"(NOMBRE_PUNTO); FECHA_INICIO de {fechas_txt[fechas_txt != 'NA'].min()} a {fechas_txt[fechas_txt != 'NA'].max()} "
      f"({(fechas_txt == 'NA').sum()} filas con «NA»)")

## 2. Cada fila larga con su destino: la cascada de Go, fila por fila

`reports/limpieza.json` dice **cuántas** filas descarta cada regla; para preguntar **por qué** hace falta
saber **cuáles**. `etiquetar` replica la cascada de `datos/bloque.go` en el mismo orden (como
`aire_lab.limpiar`, pero devuelve el estado de cada fila: `valida` o la primera regla que la descarta)
con la misma config, incluidas las tres piezas que el cargador incorporó el 01/11 a partir de este
análisis: `sin_valor` (la fila vacía de la plantilla de 2022), la hora por **bloque físico** (§5.1) y
`cero_como_faltante` (§4). Primero se comprueba que los conteos son **idénticos** a los de Go; si no, lo que
sigue no vale.

In [ ]:
CFG, ORIGEN_CFG = al.cargar_config([c / "limpieza.json" for c in CONFIGS])
REG = al.compilar(CFG)
print("config:", ORIGEN_CFG)
METEO_DIA = ["temperatura", "humedad", "viento_velocidad", "viento_direccion", "presion", "radiacion"]


def etiquetar(b: pd.DataFrame) -> pd.DataFrame:
    """Una fila por fila larga del bronze (en orden de archivo) con su parámetro, valor, día, hora
    reconstruida y `estado`: 'valida' o la primera regla de la cascada de Go que la descarta. Como Go
    desde el 01/11: sin_valor, hora por bloque físico (al.bloques_fisicos) y cero_como_faltante."""
    n = len(b)
    vacios = list(REG["vacios"])
    reales = list(REG["en_columnas"])
    con = np.column_stack([~b[r].astype(str).str.strip().isin(vacios).to_numpy() for r in reales])
    cuantos, cual = con.sum(1), con.argmax(1)
    canon = np.where(cuantos == 1, np.asarray([REG["en_columnas"][r] for r in reales], dtype=object)[cual], None)
    valor = np.full(n, np.nan)
    for j, r in enumerate(reales):
        m = (cuantos == 1) & (cual == j)
        if m.any():
            valor[m] = pd.to_numeric(b[r].astype(str)[m].str.strip(), errors="coerce").to_numpy()
    def por_cat(col, f):  # aplica f a los valores distintos y expande (rápido sobre categorías)
        cats = [str(c) for c in b[col].cat.categories]
        return np.asarray([f(c) for c in cats] + [f("")], dtype=object)[b[col].cat.codes.to_numpy()]

    est = por_cat("NOMBRE_PUNTO", al.nombre_estacion)
    per = pd.Series(por_cat("PERIODO", str.strip), index=b.index)
    per_n = por_cat("PERIODO", lambda x: al.normalizar(x.strip()))
    uni = por_cat("UNIDAD_MEDIDA", str.strip)
    fecha = por_cat("FECHA_INICIO", str.strip)
    hseg = por_cat("HORA_INICIO", lambda x: al._segundos_hora(
        x, REG["hora_prefijo"], REG["vacios"], REG["solo_prefijo_sin_hora"], REG["truncar"])).astype(float)
    explicita = ~np.isnan(hseg) & (hseg >= 0)
    tipo = por_cat("TIPO_ANALISIS", str.strip)
    dia = pd.to_datetime(pd.Series(fecha), format="%Y-%m-%d", errors="coerce").to_numpy()
    en_rango = ~np.isnat(dia) & (dia >= np.datetime64(REG["desde"])) & (dia < np.datetime64(REG["hasta"]))
    # bloques (estación, parámetro, fecha, unidad, periodo) sobre las filas con parámetro, como Go
    idx = np.flatnonzero(~pd.isna(canon))
    g = pd.DataFrame({"e": est[idx], "p": canon[idx], "f": fecha[idx], "u": uni[idx], "per": per.to_numpy()[idx], "i": idx})
    gg = g.groupby(["e", "p", "f", "u", "per"], sort=False)["i"]
    tam = gg.transform("size").to_numpy()
    span = (gg.transform("max") - gg.transform("min") + 1).to_numpy()
    posicion = np.full(n, -1)
    posicion[idx] = gg.cumcount().to_numpy()
    bloque = np.full(n, -1)
    bloque[idx] = gg.ngroup().to_numpy()
    filas_dia = int(REG["hpp"].get("filas_por_dia", 24))
    completo = np.zeros(n, bool)
    completo[idx] = (tam == filas_dia) & (span == filas_dia)
    completo_dia = completo.copy()
    # Bloque físico (= Go): si el día del parámetro no tiene 24 filas, la posición en las filas contiguas
    # de la misma (estación, fecha, unidad, periodo, tipo de análisis), con las vacías, si son 24 de un
    # solo parámetro. Llegan a ese punto las filas con periodo no descartado y fecha válida.
    alcanza = np.isin(per_n, list(REG["periodos"]) + list(REG["periodos_por_grupo"])) & en_rango
    tam_fis, pos_fis, par_fis = al.bloques_fisicos(est, dia, uni, per.to_numpy(), tipo, alcanza, canon, cuantos == 1)
    fisico = ~pd.isna(canon) & alcanza & ~completo_dia & (tam_fis == filas_dia) & (par_fis == canon)
    completo |= fisico
    posicion = np.where(fisico, pos_fis, posicion)
    estado = np.array(["valida"] * n, dtype=object)
    vivo = np.ones(n, bool)

    def descartar(regla, m):
        m = vivo & m
        estado[m] = regla
        vivo[m] = False

    horario = np.isin(per_n, list(REG["periodos"])) | (
        np.isin(per_n, list(REG["periodos_por_grupo"])) & (explicita | completo))
    descartar("periodo", ~horario)
    descartar("sin_valor", cuantos == 0)
    descartar("parametro", cuantos > 1)
    descartar("fecha", np.isnat(dia))
    descartar("fecha_fuera_de_rango", ~en_rango)
    ilegible = hseg == -1.0
    inferida = ~explicita & ~ilegible & completo
    desfase = 3600.0 if REG["hpp"].get("convencion", "fin") == "fin" else 0.0
    ajuste = (0.0 if REG["explicita_inicio"] else -3600.0) + desfase
    seg = np.where(explicita, hseg + ajuste, np.where(inferida, posicion * 3600.0 + desfase, np.nan))
    descartar("hora_no_en_punto", hseg == -2.0)
    descartar("hora", np.isnan(seg))
    descartar("valor_no_numerico", np.isnan(valor) & ~pd.isna(canon))
    descartar("centinela", np.isin(valor, np.asarray(REG["centinelas"], dtype=float)))
    pk = np.where(pd.isna(canon), "∅", canon)
    f_par = {(p, u): (1.0 if p == "∅" or p not in REG["unidades"] else REG["unidades"][p].get(al.normalizar(u), np.nan))
             for p, u in set(zip(pk, uni))}
    factor = np.array([f_par[k] for k in zip(pk, uni)], dtype=float)
    descartar("unidad", np.isnan(factor) & ~pd.isna(canon))
    neg_ok = pd.Series(canon, dtype=object).isin(list(REG["negativos"])).to_numpy()
    descartar("negativo", (valor < 0) & ~neg_ok)
    vcan = valor * np.nan_to_num(factor, nan=1.0)
    lo, hi = np.full(n, -np.inf), np.full(n, np.inf)
    for p, (a, z) in REG["rangos"].items():
        s = canon == p
        lo[s], hi[s] = a, z
    descartar("fuera_de_rango", (vcan < lo) | (vcan > hi))
    hora_fisica = vivo & inferida & fisico  # = hora_por_bloque_fisico de Go: se cuenta al recibir la hora
    cand = vivo & (bloque >= 0) & pd.Series(canon, dtype=object).isin(list(REG["bloques_en_cero"])).to_numpy()
    ii = np.flatnonzero(cand)
    todo0 = pd.Series(vcan[ii] == 0).groupby(bloque[ii]).transform("all").to_numpy()
    mz = np.zeros(n, bool)
    mz[ii[todo0]] = True
    descartar("bloque_en_cero", mz)
    descartar("cero_como_faltante", (vcan == 0) & pd.Series(canon, dtype=object).isin(list(REG["cero_faltante"])).to_numpy())
    fh = dia + np.nan_to_num(seg).astype("timedelta64[s]")
    canonica = factor == 1.0
    v = pd.DataFrame({"e": est, "p": canon, "fh": fh, "can": canonica})[vivo]
    hay = v.groupby(["e", "p", "fh"], sort=False)["can"].transform("max").to_numpy()
    m = np.zeros(n, bool)
    m[np.flatnonzero(vivo)[hay & ~v["can"].to_numpy()]] = True
    descartar("duplicado_unidad", m)
    v = pd.DataFrame({"e": est, "p": canon, "fh": fh, "x": vcan})[vivo]
    dup = v.duplicated(["e", "p", "fh"], keep="first").to_numpy()
    primero = v.groupby(["e", "p", "fh"], sort=False)["x"].transform("first").to_numpy()
    m1, m2 = np.zeros(n, bool), np.zeros(n, bool)
    m1[np.flatnonzero(vivo)[dup & (v["x"].to_numpy() == primero)]] = True
    m2[np.flatnonzero(vivo)[dup & (v["x"].to_numpy() != primero)]] = True
    descartar("duplicado_exacto", m1)
    descartar("duplicado_valor_distinto", m2)
    return pd.DataFrame({
        "estacion": pd.Categorical(est), "evaluacion": b["NOMBRE_EVALUACION"], "tipo": b["TIPO_ANALISIS"],
        "periodo": b["PERIODO"], "unidad": b["UNIDAD_MEDIDA"], "parametro": pd.Categorical(canon),
        "valor": valor, "valor_canon": vcan, "dia": dia, "fecha_hora": fh, "hora_explicita": explicita,
        "hora_txt": b["HORA_INICIO"], "posicion": posicion, "bloque": bloque, "completo": completo,
        "completo_dia": completo_dia, "tam_fisico": tam_fis, "hora_fisica": hora_fisica,
        "estado": pd.Categorical(estado)})


t0 = time.time()
L = etiquetar(B)
print(f"etiquetadas {len(L):,} filas en {time.time() - t0:.0f} s")
conteo = L["estado"].value_counts()
ruta_go = REPORTS / "limpieza.json"
GO = json.loads(ruta_go.read_text(encoding="utf-8")) if ruta_go.exists() else None
filas = [{"regla": "valida (procesadas)", "python": int(conteo.get("valida", 0)),
          "go": GO["filas_largas"]["procesadas"] if GO else None}]
for r in al.NOMBRES_REGLAS:
    filas.append({"regla": r, "python": int(conteo.get(r, 0)),
                  "go": GO["filas_largas"]["descartadas_por_regla"].get(r) if GO else None})
filas.append({"regla": "(hora por bloque físico, no es un descarte)", "python": int(L["hora_fisica"].sum()),
              "go": GO["filas_largas"].get("hora_por_bloque_fisico") if GO else None})
cuadre = pd.DataFrame(filas)
cuadre["igual"] = cuadre["python"] == cuadre["go"]
tabla(cuadre[(cuadre.python > 0) | (cuadre.go.fillna(0) > 0)])
if GO is None:
    print("AVISO: no hay reports/limpieza.json de Go: no se puede cuadrar (lo que sigue usa la réplica).")
elif not cuadre["igual"].all():
    print("AVISO: la réplica no cuadra con Go en alguna regla: revisar antes de leer lo que sigue.")
else:
    print("La réplica cuadra con Go en todas las reglas: cada fila tiene la etiqueta que le pone Go.")

### 2.1 Dos extras para leer las filas: a qué parámetro pertenece una fila vacía y en qué formato vino la exportación

- **Parámetro de una fila vacía.** Una fila sin ningún valor no dice de qué parámetro es. Pero el archivo
  está ordenado: las filas de una misma serie (estación, unidad, periodo, tipo de análisis) van contiguas.
  Si en un tramo contiguo todas las filas con valor son del mismo parámetro, las vacías del tramo se le
  asignan (**inferencia** por posición; si el tramo no tiene ninguna fila con valor, queda «¿?» con su unidad).
- **Formato de exportación.** Se cuenta cuántos tramos contiguos de (parámetro, unidad, periodo) tiene cada
  estación (mirando solo sus propias filas, en orden de archivo) por día con filas: 10 o más por día es una
  exportación **día a día** (las 24 horas de cada parámetro, día tras día); menos de 1 por día es una
  exportación **por serie** (todo un parámetro de corrido, luego el siguiente).

In [ ]:
L["per"] = L["periodo"].astype(str).str.strip().str.lower()
L["p"] = L["parametro"].astype(object).where(L["parametro"].notna(), "(ninguno)").astype(str)
L["u"] = L["unidad"].astype(str)
L["anio"] = pd.DatetimeIndex(L["dia"]).year
L["es0"] = L["valor"] == 0
L["no_nula"] = L["valor"].notna() & (L["valor"] != 0)
L["valor_no_nulo"] = L["valor"].where(L["no_nula"])
HORARIOS = {"1 hora", "-"}
L["horaria"] = L["per"].isin(HORARIOS)

# tramos contiguos de una serie (estación, unidad, periodo, tipo) en orden de archivo
serie = L.groupby(["estacion", "unidad", "periodo", "tipo"], observed=True, sort=False, dropna=False).ngroup()
tramo = (serie != serie.shift()).cumsum()
con_valor = L[L["p"] != "(ninguno)"]
params_tramo = con_valor.groupby(tramo[con_valor.index])["p"].agg(lambda s: "|".join(sorted(set(s))))
L["p_inf"] = L["p"]
vacias = L["p"] == "(ninguno)"
asignado = tramo[vacias].map(params_tramo)
POR_UNIDAD = {"mmHg": "presion", "hPa": "presion", "mm": "precipitacion", "%": "humedad", "m/s": "viento_velocidad",
              "W/m2": "radiacion"}  # unidades que solo usa un parámetro; °c, ug/m3 y ppb son ambiguas
por_unidad = L.loc[vacias, "u"].map(POR_UNIDAD)
L.loc[vacias, "p_inf"] = np.where(asignado.notna(), asignado.fillna(""),
                                  np.where(por_unidad.notna(), por_unidad.fillna(""), "¿? [" + L.loc[vacias, "u"] + "]"))
L.loc[vacias & L["p_inf"].str.contains(r"\|", regex=True), "p_inf"] = "¿? varios"
print("filas vacías con parámetro inferido:", f"{(vacias & ~L['p_inf'].str.startswith('¿')).sum():,}",
      "de", f"{vacias.sum():,}")

# formato de exportación por estación
pu = L.groupby(["estacion", "parametro", "unidad", "periodo"], observed=True, sort=False, dropna=False).ngroup()
anterior = pu.groupby(L["estacion"], observed=True).shift()  # la fila anterior DE LA MISMA estación
tramos_pu = (pu != anterior).groupby(L["estacion"], observed=True).sum()
dias_est = L.groupby("estacion", observed=True)["dia"].nunique()
FORMATO = pd.DataFrame({"tramos": tramos_pu, "dias": dias_est})
FORMATO["tramos_por_dia"] = FORMATO.tramos / FORMATO.dias
FORMATO["formato"] = np.where(FORMATO.tramos_por_dia >= 3, "día a día", "por serie")
FORMATO["pct_hora_explicita"] = 100 * L.groupby("estacion", observed=True)["hora_explicita"].mean()
FORMATO["pct_filas_vacias"] = 100 * vacias.groupby(L["estacion"], observed=True).mean()
tabla(FORMATO.reset_index().sort_values(["formato", "estacion"]), decimales=2)

### 2.2 El archivo es una concatenación de exportaciones, una por evaluación

Se recorren las filas en orden y se cuentan los tramos contiguos de una misma `NOMBRE_EVALUACION`.

In [ ]:
ev = L["evaluacion"].astype(str)
corte = (ev != ev.shift()).cumsum()
TRAMOS_EV = L.groupby(corte.rename("tramo")).agg(evaluacion=("evaluacion", "first"), filas=("estacion", "size"),
                                 estaciones=("estacion", lambda s: ", ".join(sorted(set(map(str, s))))),
                                 desde=("dia", "min"), hasta=("dia", "max"))
print(f"{len(TRAMOS_EV)} tramos contiguos para {ev.nunique()} evaluaciones")
resumen_ev = TRAMOS_EV.groupby("evaluacion", sort=False).agg(
    tramos=("filas", "size"), filas=("filas", "sum"),
    estaciones=("estaciones", lambda s: ", ".join(sorted(set(", ".join(s).split(", "))))),
    desde=("desde", "min"), hasta=("hasta", "max")).reset_index()
resumen_ev["desde"] = resumen_ev["desde"].dt.date
resumen_ev["hasta"] = resumen_ev["hasta"].dt.date
tabla(resumen_ev)

## 3. (a) ¿Por qué falta la hora? ¿Es por estación, por periodo o por parámetro?

In [ ]:
L["hora_tipo"] = np.select(
    [L["hora_txt"].astype(str) == "NA", L["hora_txt"].astype(str).str.fullmatch(r"1899-12-31 \d\d:\d\d:\d\d"),
     L["hora_txt"].astype(str) == "1899-12-31"],
    ["NA", "1899-12-31 HH:MM:SS", "1899-12-31 (solo)"], "otro")
print(L["hora_tipo"].value_counts().to_string())
por_est = pd.crosstab(L["estacion"], L["hora_tipo"])
por_est["% con hora"] = 100 * (1 - por_est["NA"] / por_est.sum(axis=1))
md("**Por estación** (solo las que tienen alguna hora explícita):")
tabla(por_est[por_est["% con hora"] > 0].reset_index(), decimales=2)
por_mes = L.groupby(pd.DatetimeIndex(L["dia"]).to_period("M"))["hora_explicita"].agg(["size", "mean"])
desde_hora = por_mes[por_mes["mean"] > 0].index.min()
md(f"**Por mes**: la primera fila con hora explícita es de **{desde_hora}**; antes, 0 %.")
d = L[L["dia"] >= pd.Timestamp("2021-12-01")]
otras = d[~d["estacion"].isin(["CA-HUA-01", "CA-MAC-01"])]
print(f"Desde 2021-12, fuera de CA-HUA-01 y CA-MAC-01: {otras['hora_explicita'].mean():.3%} de filas con hora "
      f"({otras['hora_explicita'].sum()} de {len(otras):,}), en {otras['estacion'].nunique()} estaciones.")
ag = L[L["estacion"].isin(["CA-HUA-01", "CA-MAC-01"])]
md("**Por parámetro**, en las dos estaciones con hora (Agroaurora): la hora viene en todos los parámetros.")
tabla(pd.crosstab(ag["p_inf"], ag["hora_tipo"]).reset_index())

**Control de la hora por posición.** En los bloques completos que traen hora, ¿la hora explícita es la
posición de la fila en el bloque? Y «1899-12-31» sin hora, ¿es la posición 0 (00:00)?

In [ ]:
c = L[L["hora_explicita"] & L["completo"]]
hh = (c["fecha_hora"] - c["dia"]).dt.total_seconds() / 3600 - 1  # deshace la convención «fin»: hora de inicio
print(f"filas con hora en bloques completos: {len(c):,}; hora de inicio == posición: {(hh == c['posicion']).mean():.2%}")
solo = L[(L["hora_tipo"] == "1899-12-31 (solo)") & L["completo"]]
print(f"«1899-12-31» solo, en bloques completos: {len(solo):,} filas; en la posición 0: {(solo['posicion'] == 0).mean():.2%}")

**Control externo con un informe de OEFA.** El REPORTE N° 00011-2021-OEFA/DEAM-STEC (EAS de la UM Antamina,
diciembre de 2021; ver `docs/eda-procedencia.md`) dice que el monitoreo continuo de PM en CA-AYAS-01
empezó el 9/12 a las 14:35 y que hubo un corte de luz del 10/12 15:45 al 11/12 15:05, y publica los
promedios de 24 h desde el 12/12. CA-AYAS-01 no trae ninguna hora: si la hora por posición es correcta,
el 10/12 tiene que quedar en cero desde la posición 16 (16:00) y el 11/12 volver en la posición 15 (15:00).

In [ ]:
ay = L[(L["estacion"] == "CA-AYAS-01") & (L["p"] == "pm2_5") & L["dia"].between("2021-12-09", "2021-12-13")]
for d_, g in ay.groupby("dia"):
    h = g[g["horaria"]]
    ceros = h.loc[h["valor"] == 0, "posicion"].tolist() if h["completo"].any() else "(día incompleto)"
    v24 = g.loc[~g["horaria"], "valor"].tolist()
    print(f"{d_.date()}: {len(h):2d} filas horarias, posiciones en 0: {ceros}; 24 h = {v24}")

## 4. (b) Los días enteros en cero: ¿dónde, cuándo, de qué largo?

Un **día en cero** es un bloque (estación, parámetro, día, unidad, periodo horario) cuyas lecturas son
todas 0 exacto. Se cuentan también los de presión, que Go descarta antes por `fuera_de_rango` (0 mmHg está
fuera de [400, 1100] hPa) y que son el mismo fenómeno. La precipitación queda fuera: un día seco es 0.

In [ ]:
h = L[L["horaria"] & L["parametro"].notna() & (L["p"] != "precipitacion") & L["dia"].notna()]
BLK = h.groupby("bloque").agg(estacion=("estacion", "first"), p=("p", "first"), dia=("dia", "first"),
                               n=("valor", "size"), cero=("es0", "min"))
dia_p = BLK.groupby(["estacion", "p", "dia"], observed=True)["cero"].all().reset_index()
print(f"bloques horarios: {len(BLK):,}; en cero: {BLK['cero'].sum():,} "
      f"({BLK.loc[BLK['cero'], 'n'].sum():,} lecturas, de ellas "
      f"{len(L[(L['estado'] == 'fuera_de_rango') & (L['p'] == 'presion') & (L['valor'] == 0)]):,} de presión que Go cuenta en fuera_de_rango)")
frac = dia_p.groupby(["estacion", "p"], observed=True)["cero"].mean().unstack("p")
md("**Fracción de días en cero por estación y parámetro** (1.00 = el parámetro nunca tuvo un día con valor):")
tabla((frac.round(2)).reset_index(), decimales=2)


def tramos_en_cero(dp: pd.DataFrame) -> pd.DataFrame:
    """Tramos de días consecutivos en cero por (estación, parámetro)."""
    out = []
    for (e, p), g in dp.groupby(["estacion", "p"], observed=True):
        g = g.sort_values("dia")
        nuevo = (~g["cero"]) | (g["dia"].diff() != pd.Timedelta(days=1)) | (~g["cero"].shift(fill_value=False))
        g = g.assign(t=nuevo.cumsum())
        primero, ultimo, total = g["dia"].min(), g["dia"].max(), len(g)
        for _, t in g[g["cero"]].groupby("t"):
            ini, fin = t["dia"].min(), t["dia"].max()
            out.append({"estacion": e, "parametro": p, "inicio": ini.date(), "fin": fin.date(), "dias": len(t),
                        "dias_del_parametro": total, "al_inicio": ini == primero, "al_final": fin == ultimo})
    t = pd.DataFrame(out)
    t["tipo"] = np.select([t["dias"] >= 0.95 * t["dias_del_parametro"], t["dias"] > 7, t["dias"] > 1],
                          ["nunca medido", "tramo largo (> 7 días)", "tramo corto (2–7 días)"], "día suelto")
    return t


TRAMOS0 = tramos_en_cero(dia_p)
TRAMOS0.to_csv(REPORTS / "faltantes_tramos_en_cero.csv", index=False)
res = TRAMOS0.groupby("tipo").agg(tramos=("dias", "size"), dias_parametro=("dias", "sum"))
res["% de los días en cero"] = 100 * res["dias_parametro"] / res["dias_parametro"].sum()
tabla(res.reset_index())
largos = TRAMOS0[TRAMOS0["tipo"] != "nunca medido"]
print(f"tramos (sin contar 'nunca medido'): {len(largos)}; al inicio de la serie del parámetro: "
      f"{largos['al_inicio'].sum()}, al final: {largos['al_final'].sum()}; días en esos tramos: "
      f"{largos['dias'].sum():,}, de ellos en tramos de inicio o fin: {largos.loc[largos.al_inicio | largos.al_final, 'dias'].sum():,}")
nunca = TRAMOS0[TRAMOS0["tipo"] == "nunca medido"].groupby("estacion")["parametro"].agg(", ".join)
md("**Parámetros que una estación «trae» pero nunca midió** (todas o casi todas sus filas en 0):")
tabla(nunca.reset_index())

In [ ]:
# ¿Se apaga la estación entera o un sensor? Días en que todos los bloques horarios (sin precipitación) están en cero.
ed = dia_p.groupby(["estacion", "dia"], observed=True)["cero"].agg(["size", "sum"])
ed["toda"] = ed["sum"] == ed["size"]
ed["alguno"] = ed["sum"] > 0
est_dias = ed.groupby(level=0).agg(dias=("size", "size"), toda_en_cero=("toda", "sum"), algun_parametro=("alguno", "sum"))
md("**Días con la estación entera en cero** frente a días con algún parámetro en cero:")
tabla(est_dias[est_dias.algun_parametro > 0].reset_index())
# Meteorología: ¿caen juntas temperatura, humedad, viento y presión? (un datalogger apagado)
met = dia_p[dia_p["p"].isin(["temperatura", "humedad", "viento_velocidad", "viento_direccion"])]
mm = met.groupby(["estacion", "dia"], observed=True)["cero"].agg(["size", "sum"])
mm = mm[mm["size"] == 4]
print(f"días con las 4 variables de meteorología: {len(mm):,}; con alguna en cero: {(mm['sum'] > 0).sum():,}; "
      f"de esos, con las 4 en cero a la vez: {(mm['sum'] == 4).sum() / max((mm['sum'] > 0).sum(), 1):.1%}")
# Por año: cuánto falta como cero y cuánto como fila vacía (los dos formatos de exportación)
hv = L[L["horaria"] & (L["p_inf"] != "precipitacion") & L["dia"].notna()]
cel = hv.groupby(["estacion", "p_inf", "dia"], observed=True).agg(
    n=("valor", "size"), con_valor=("valor", "count"), ceros=("es0", "sum"))
cel["estado"] = np.select([cel["con_valor"] == 0, cel["ceros"] == cel["con_valor"]], ["vacío", "en cero"], "con datos")
anio_cel = pd.Series(cel.index.get_level_values("dia").year.astype(str), index=cel.index, name="año")
anual = pd.crosstab(anio_cel, cel["estado"], normalize="index") * 100
md("**Días-parámetro horarios por año y estado** (%): hasta 2021 lo que falta viene como **cero**; en 2022, como **fila vacía**.")
tabla(anual.round(1).reset_index())

**Ceros sueltos dentro de días con datos.** `bloque_en_cero` solo saca días enteros. Hasta el 01/11 las
horas en 0 dentro de un día con datos (el corte de luz de CA-AYAS-01) pasaban como lecturas válidas; con
este análisis (y la regla de las 18 h de §7: OEFA no promedia las horas en 0), el cargador las descarta
ahora con `cero_como_faltante` en los contaminantes (PM2.5, PM10 y gases). La meteorología queda: un
viento de 0 m/s es calma. La tabla cuenta, por parámetro, los ceros entre las lecturas que pasan las
reglas de valor: los que descarta `cero_como_faltante` y los que siguen como válidos.

In [ ]:
v = L[L["estado"].isin(["valida", "cero_como_faltante"])]
z = v.groupby("p").agg(lecturas=("valor", "size"), ceros=("valor", lambda x: int((x == 0).sum())))
z["% en cero"] = 100 * z["ceros"] / z["lecturas"]
z["descartados por cero_como_faltante"] = L[L["estado"] == "cero_como_faltante"].groupby("p").size().reindex(z.index, fill_value=0)
tabla(z.reset_index(), decimales=2)
pm = v[v["p"].isin(["pm10", "pm2_5"])].sort_values(["estacion", "p", "fecha_hora"])
es0 = pm["valor"] == 0
corrida = ((es0 != es0.shift()) | (pm["estacion"] != pm["estacion"].shift()) | (pm["p"] != pm["p"].shift())).cumsum()
largos0 = pm[es0].groupby(corrida[es0]).size()
print(f"PM en 0 dentro de días con datos (hoy los descarta cero_como_faltante): {es0.sum():,} lecturas en "
      f"{len(largos0):,} corridas de horas (mediana {largos0.median():.0f} h, máx {largos0.max()} h)")
ruta_silver = DATA / "silver" / "oefa_ancho.csv"
if ruta_silver.exists():
    S = pd.read_csv(ruta_silver, usecols=["estacion", "fecha_hora", "pm2_5", "pm10", "temperatura", "humedad", "hora_inferida"])
    amb = S.dropna(subset=["pm2_5", "pm10"])
    print(f"silver (Go): {len(S):,} filas anchas; con PM2.5 y PM10: {len(amb):,}; con los dos en 0: "
          f"{((amb.pm2_5 == 0) & (amb.pm10 == 0)).sum():,}")
else:
    S = None
    print("No hay silver de Go: se omite la cuenta en el silver.")

## 5. (c) Las filas sin ningún valor: ¿qué son?

In [ ]:
sv = L[vacias]
print(f"filas sin ningún valor: {len(sv):,}; de 2022: {(sv['anio'] == 2022).sum():,} ({(sv['anio'] == 2022).mean():.2%})")
tabla(sv.groupby(["tipo", "periodo", "unidad"], observed=True).size().rename("filas").reset_index())
md("**Por estación y año**:")
tabla(pd.crosstab(sv["estacion"], sv["anio"].astype("Int64").astype(str).rename("año")).reset_index())
celv = (L[L["horaria"]].assign(vacia=vacias).groupby(["estacion", "p_inf", "dia"], observed=True)["vacia"]
        .agg(["sum", "size"]).rename(columns={"sum": "vacias", "size": "filas_dia"}))
celv = celv[celv["vacias"] > 0].reset_index()
celv["entero"] = celv["vacias"] == celv["filas_dia"]  # todas las filas horarias del día (en todas sus unidades) vacías
SINV = (celv.groupby(["estacion", "p_inf"], observed=True)
        .agg(filas=("vacias", "sum"), dias=("vacias", "size"), dias_enteros=("entero", "sum"),
             desde=("dia", "min"), hasta=("dia", "max")).reset_index())
SINV["desde"], SINV["hasta"] = SINV["desde"].dt.date, SINV["hasta"].dt.date
total_pi = L[L["horaria"]].groupby(["estacion", "p_inf"], observed=True)["dia"].nunique().rename("dias_del_parametro")
SINV = SINV.merge(total_pi.reset_index(), on=["estacion", "p_inf"], how="left")
SINV["% de los días del parámetro"] = (100 * SINV["dias"] / SINV["dias_del_parametro"]).round(2)
SINV = SINV.rename(columns={"p_inf": "parametro_inferido"})
SINV.to_csv(REPORTS / "faltantes_sin_valor.csv", index=False)
print(f"de los días-parámetro con alguna fila vacía, enteros (todas sus filas horarias vacías): {SINV['dias_enteros'].sum() / SINV['dias'].sum():.1%}")
tabla(SINV.sort_values("filas", ascending=False).head(25))

### 5.1 La hora de los días con filas vacías: el bloque físico (y un tercer formato)

Hasta el 01/11, Go reconstruía la hora con la posición de la fila **dentro de las filas con valor** de su
día (estación, parámetro, día, unidad, periodo) y exigía 24 contiguas. En la plantilla de 2022 un hueco es
una fila vacía que **ocupa su hora**: un día con 23 lecturas y 1 vacía tenía un bloque de 23, y Go tiraba
las 23 por la regla `hora`. Prueba de que la posición dentro del **bloque físico** (las filas contiguas
del día, con o sin valor) es la hora: en CA-HUA-01 y CA-MAC-01, que traen hora explícita. Con esta
prueba, el cargador cuenta ahora la posición en el bloque físico cuando el día del parámetro no tiene 24
filas (si el bloque tiene 24 de un solo parámetro) y recupera esas lecturas (`hora_por_bloque_fisico` en
`reports/limpieza.json`). Lo que queda en la regla `hora` son días que empiezan o terminan a media jornada
y el formato intercalado hora a hora de CA-CH-03.

In [ ]:
k_fis = L.groupby(["estacion", "unidad", "periodo", "tipo", "dia"], observed=True, sort=False, dropna=False).ngroup()
fisico = (k_fis != k_fis.shift()).cumsum()
tam_fis = fisico.map(fisico.value_counts())
pos_fis = L.groupby(fisico).cumcount()
ag = L[L["estacion"].isin(["CA-HUA-01", "CA-MAC-01"]) & L["hora_explicita"] & L["horaria"] & (tam_fis == 24)]
h_ini = (ag["fecha_hora"] - ag["dia"]).dt.total_seconds() / 3600 - 1
inc = ~ag["completo"]
print(f"Agroaurora, filas con hora en bloques físicos de 24: {len(ag):,}; hora == posición física: {(h_ini == pos_fis[ag.index]).mean():.2%}")
print(f"  de ellas en bloques de parámetro con huecos (sin hora, Go las tiraría): {inc.sum():,}; "
      f"hora == posición física: {(h_ini[inc] == pos_fis[ag.index][inc]).mean():.2%}")
print(f"lecturas con la hora de su bloque físico (las que antes tiraba la regla hora; Go: hora_por_bloque_fisico): "
      f"{int(L['hora_fisica'].sum()):,}; quedan en la regla hora: {int((L['estado'] == 'hora').sum()):,}")
x = L[L["estado"] == "hora"].assign(tam=L["tam_fisico"], tam_param=L["bloque"].map(L.loc[L["bloque"] >= 0, "bloque"].value_counts()))
x["caso"] = np.select([x["tam"] == 24, x["tam_param"] == 24],
                      ["bloque físico de 24 filas con más de un parámetro", "24 filas del parámetro intercaladas hora a hora"],
                      "día incompleto (empieza o termina a media jornada)")
PERDIDA_HORA = pd.crosstab(x["estacion"], x["caso"])
tabla(PERDIDA_HORA.reset_index())
print({k: int(v) for k, v in x["caso"].value_counts().items()})
inter = L[(L["estacion"] == "CA-CH-03") & (L["estado"].isin(["hora", "periodo"])) & L["horaria"] & ~vacias]
print(f"CA-CH-03 en formato hora a hora: {inter['dia'].min().date()} a {inter['dia'].max().date()}, "
      f"{inter['dia'].nunique()} días, {len(inter):,} lecturas descartadas (hora + periodo «-»)")

## 6. (d) Duplicados: ¿qué se repite y por qué?

In [ ]:
ident = B.duplicated(keep="first")
print(f"filas idénticas a otra anterior en las 36 columnas: {ident.sum():,} ({ident.mean():.1%})")
print(f"  de esas, con hora NA: {(ident & (L['hora_tipo'] == 'NA')).sum():,}; con hora explícita: "
      f"{(ident & (L['hora_tipo'] != 'NA')).sum():,}")
print(f"  de esas, en un día en cero o vacías: {(ident & (L['estado'].isin(['bloque_en_cero', 'sin_valor']) | ((L['estado'] == 'fuera_de_rango') & (L['valor'] == 0)))).sum():,}")
print(f"con la hora reconstruida, duplicados exactos de (estación, parámetro, fecha-hora): "
      f"{int(conteo.get('duplicado_exacto', 0))}; con otro valor: {int(conteo.get('duplicado_valor_distinto', 0))}")

In [ ]:
du = L[L["estado"] == "duplicado_unidad"]
md("**La misma lectura en dos unidades** (filas que Go descarta por `duplicado_unidad`, por estación y parámetro):")
tabla(pd.crosstab(du["estacion"], du["p"]).reset_index())
gem = L[L["estado"].isin(["valida", "duplicado_unidad"]) & L["p"].isin(["so2", "co", "o3", "h2s", "presion"])]
piv = gem.pivot_table(index=["estacion", "p", "fecha_hora"], columns=gem["estado"].astype(str), values="valor_canon",
                      aggfunc="first", observed=True).dropna()
piv = piv[(piv["valida"] > 0) & (piv["duplicado_unidad"] > 0)]
piv["cociente"] = piv["valida"] / piv["duplicado_unidad"]
tabla(piv.groupby(level="p")["cociente"].describe(percentiles=[.05, .5, .95])[["count", "5%", "50%", "95%"]].reset_index(), decimales=3)
con_gas = L[L["p"].isin(["so2", "co", "o3", "h2s"]) & L["horaria"]]
ugas = con_gas.groupby(["estacion", "p"], observed=True)["u"].agg(lambda s: "+".join(sorted(set(s))))
print("gases por estación y unidades en que vienen:", ugas.value_counts().to_dict())
pres = L[(L["p"] == "presion") & L["horaria"]].groupby("estacion", observed=True)["u"].agg(lambda s: "+".join(sorted(set(s))))
print("presión por estación:", pres.value_counts().to_dict(), "->", ", ".join(pres[pres.str.contains(r"\+")].index))

In [ ]:
md("**Otras repeticiones que no son duplicados de lecturas**:")
ren = L.groupby(["estacion", "evaluacion"], observed=True)["dia"].agg(["size", "min", "max"]).reset_index()
multi = ren[ren.duplicated("estacion", keep=False)].copy()
multi["min"], multi["max"] = multi["min"].dt.date, multi["max"].dt.date
md("- Estaciones con más de un `NOMBRE_EVALUACION` (¿se solapan en el tiempo?):")
tabla(multi.rename(columns={"size": "filas", "min": "desde", "max": "hasta"}))
ub = B.groupby("NOMBRE_PUNTO", observed=True)[["ESTE", "NORTE", "DESCRIPCION_UBICACION"]].agg(lambda s: s.value_counts().index[0])
mismo = ub[ub.duplicated(["ESTE", "NORTE"], keep=False)]
md("- Estaciones distintas con las mismas coordenadas:")
rango = L.groupby("estacion", observed=True)["dia"].agg(desde="min", hasta="max").reset_index()
rango["desde"], rango["hasta"] = rango["desde"].dt.date, rango["hasta"].dt.date
rango["estacion"] = rango["estacion"].astype(str)
tabla(mismo.reset_index().assign(NOMBRE_PUNTO=lambda d: d["NOMBRE_PUNTO"].astype(str))
      .merge(rango, left_on="NOMBRE_PUNTO", right_on="estacion").drop(columns="estacion"))
md("- Variantes de escritura de un mismo valor:")
for c in ["PERIODO", "PROCEDENCIA_MUESTRA"]:
    print(f"  {c}: {B[c].value_counts().to_dict()}")
alt = B.groupby("NOMBRE_PUNTO", observed=True)["ALTITUD"].nunique()
desc = B.groupby("NOMBRE_PUNTO", observed=True)["DESCRIPCION_UBICACION"].nunique()
print("  estaciones con más de una ALTITUD:", alt[alt > 1].to_dict())
print("  estaciones con más de una DESCRIPCION_UBICACION (espacios, saltos de línea, redacción):", desc[desc > 1].to_dict())

## 7. (e) Promedios de 24 h y móviles: ¿además de los horarios o en lugar de ellos?

In [ ]:
nh = L[~L["horaria"]]
md("**Filas no horarias por periodo, parámetro y unidad**:")
tabla(pd.crosstab([nh["periodo"].astype(str), nh["p"]], nh["u"]).reset_index())
hor = L[L["horaria"] & L["valor"].notna()]
hm = hor.groupby(["estacion", "p", "u", "dia"], observed=True).agg(
    n=("valor", "size"), no_nulas=("no_nula", "sum"), media=("valor", "mean"), media_no_nula=("valor_no_nulo", "mean"))
d24 = L[(L["per"] == "24 horas") & L["valor"].notna()].set_index(["estacion", "p", "u", "dia"])[["valor"]].join(hm, how="left")
print(f"filas de 24 h con valor: {len(d24):,}; con serie horaria el mismo día: {d24['n'].notna().sum():,} "
      f"({d24['n'].notna().mean():.1%}); sin horario: {d24['n'].isna().sum():,} en "
      f"{d24[d24['n'].isna()].index.get_level_values(0).nunique()} estaciones")
c24 = d24[d24["n"] == 24].copy()
rel = lambda a, b: (a - b).abs() / b.abs().clip(lower=0.1)
c24["24h_en_cero"] = c24["valor"] == 0
c24["24h_igual_media_no_nulas"] = rel(c24["valor"], c24["media_no_nula"]) < 0.01
c24["horas_no_nulas"] = pd.cut(c24["no_nulas"], [-1, 0, 6, 12, 17, 18, 23, 24],
                               labels=["0", "1–6", "7–12", "13–17", "18", "19–23", "24"])
REGLA18 = c24.groupby("horas_no_nulas", observed=True).agg(
    dias=("valor", "size"), pct_24h_en_cero=("24h_en_cero", "mean"),
    pct_24h_igual_media=("24h_igual_media_no_nulas", "mean")).reset_index()
REGLA18[["pct_24h_en_cero", "pct_24h_igual_media"]] = (100 * REGLA18[["pct_24h_en_cero", "pct_24h_igual_media"]]).round(2)
REGLA18.to_csv(REPORTS / "faltantes_promedio_24h.csv", index=False)
md("**El promedio de 24 h que publica OEFA frente a las 24 horas del mismo día**: con menos de 18 horas no nulas el "
   "24 h vale 0; con 18 o más, es la media de las horas no nulas. Es la regla del 75 % (18 de 24 h).")
tabla(REGLA18)

In [ ]:
def coincide_movil(per, ventana, par):
    m = L[(L["per"] == per) & (L["p"] == par) & L["valor"].notna()]
    out = []
    for (e, u), g in m.groupby(["estacion", "u"], observed=True):
        hh_ = hor[(hor["estacion"] == e) & (hor["p"] == par) & (hor["u"] == u)].sort_values(["dia", "posicion"])
        if hh_.empty:
            continue
        rod = hh_.assign(r=hh_["valor"].rolling(ventana, min_periods=ventana).mean()).set_index(["dia", "posicion"])["r"]
        g = g.assign(pos=g.groupby("dia").cumcount()).set_index(["dia", "pos"])["valor"]
        j = pd.concat([g.rename("movil"), rod.rename("rodante")], axis=1, join="inner").dropna()
        j = j[j["rodante"] != 0]
        out.append({"periodo": per, "parametro": par, "estacion": e, "unidad": u, "comparadas": len(j),
                    "% igual a la media móvil horaria": 100 * (rel(j["movil"], j["rodante"]) < 0.01).mean()})
    return out


mov = pd.DataFrame(coincide_movil("3 horas móviles", 3, "so2") + coincide_movil("móvil 3 horas", 3, "so2")
                   + coincide_movil("móvil 8 horas", 8, "co"))
md("**Los promedios móviles se calculan de la serie horaria** (no son otra medición):")
tabla(mov)

## 8. (f) Las 8 estaciones sin ninguna lectura horaria válida: ¿qué traen?

In [ ]:
validas_est = L[L["estado"] == "valida"].groupby("estacion", observed=True).size()
sin_h = sorted(set(L["estacion"].astype(str)) - set(validas_est[validas_est > 0].index.astype(str)))
print(len(sin_h), "estaciones:", sin_h)
x = L[L["estacion"].isin(sin_h + ["CA-VMP-2A"])]


def paso_mediano(s):
    d_ = pd.Series(sorted(s.dropna().unique()))
    return d_.diff().dt.days.median() if len(d_) > 1 else np.nan


SIN_H = x.groupby(["estacion", "periodo", "p"], observed=True).agg(
    filas=("valor", "size"), con_valor=("valor", "count"), en_cero=("valor", lambda s: int((s == 0).sum())),
    dias=("dia", "nunique"), desde=("dia", "min"), hasta=("dia", "max"),
    dias_entre_muestras=("dia", paso_mediano), mediana=("valor", "median")).reset_index()
SIN_H["desde"], SIN_H["hasta"] = SIN_H["desde"].dt.date, SIN_H["hasta"].dt.date
tabla(SIN_H)
tabla(B[B["NOMBRE_PUNTO"].isin(sin_h + ["CA-VMP-2A"])].groupby("NOMBRE_PUNTO", observed=True)[["TIPO_MUESTRA", "DESCRIPCION_UBICACION"]]
      .agg(lambda s: " ".join(str(s.value_counts().index[0]).split())).reset_index())

## 9. Ficha por estación

Una fila por estación con lo que el mapa y el informe necesitan. La ubicación es la combinación más
frecuente de sus columnas (como `reports/limpieza.json`); las coordenadas UTM WGS84 se pasan a lat/lon con
`aire_lab.utm_a_latlon`. **Corrección declarada**: CA-HUAR-01 trae `ZONA` 19 y departamento `NA`; su
`DESCRIPCION_UBICACION` la pone en la Comunidad Campesina Huaraucaca, junto a la planta de la UM
Colquijirca (Pasco, zona 18), y con zona 19 caería a cientos de km. Se usa zona 18 y se anota. La
corrección está en `configs/limpieza.json` (`correcciones_ubicacion`) y la aplica también el cargador de
Go, que deja el valor del CSV en `ubicacion_original` de `reports/limpieza.json`; aquí se lee de la misma
config.

In [ ]:
CORRECCION_ZONA = {e: c["zona"] for e, c in REG["correcciones"].items() if "zona" in c}  # = configs/limpieza.json


def objeto_fiscalizado(nombre: str) -> str:
    t = " ".join(str(nombre).split()).rstrip(".,")
    t = re.sub(r"^Calidad (del|de) aire (realizada )?en (el|la) ((ámbito|área) (de influencia )?(de |del )?(la |el )?)?", "", t)
    t = re.sub(r"^(ámbito|área) (de influencia )?(de |del )?(la |el )?", "", t)
    return t[:1].upper() + t[1:]


TIPOS_ACTIVIDAD = [  # solo lo que dice el nombre; si no lo dice, no se adivina
    (r"unidad minera|Minera", "minería"),
    (r"refinería de zinc|complejo metalúrgico", "metalurgia (refinación o fundición)"),
    (r"Refinería Talara", "refinería"),
    (r"agroindustrial|Casa Grande", "agroindustria"),  # Casa Grande: otra evaluación de la misma empresa dice «agroindustrial»
    (r"corredor vial", "corredor vial"),
    (r"zona ladrillera", "ladrilleras"),
    (r"zona industrial", "zona industrial"),
]


def tipo_actividad(nombre: str) -> str:
    for patron, tipo in TIPOS_ACTIVIDAD:
        if re.search(patron, str(nombre)):
            return tipo
    return "no lo dice el nombre"


cols_u = ["DEPARTAMENTO", "PROVINCIA", "DISTRITO", "UBIGEO", "ESTE", "NORTE", "ALTITUD", "ZONA", "DESCRIPCION_UBICACION"]
ubic = (B.groupby(["NOMBRE_PUNTO"] + cols_u, observed=True).size().rename("filas").reset_index()
        .sort_values(["NOMBRE_PUNTO", "filas"], ascending=[True, False]))
F = ubic.drop_duplicates("NOMBRE_PUNTO").set_index("NOMBRE_PUNTO")
F.index = F.index.astype(str)
F.index.name = "estacion"
F = F.astype({c: str for c in cols_u})
nota = {}
for e, g in ubic.groupby("NOMBRE_PUNTO", observed=True):
    partes = []
    for c in ["ESTE", "NORTE", "ALTITUD", "ZONA", "DEPARTAMENTO"]:
        vals = g.groupby(c, observed=True)["filas"].sum()
        if len(vals) > 1:
            partes.append(f"{c} con {len(vals)} valores ({', '.join(f'{k}: {al.miles(v)} filas' for k, v in vals.items())})")
    if e in CORRECCION_ZONA:
        partes.append(f"ZONA {F.loc[e, 'ZONA']} corregida a {CORRECCION_ZONA[e]} por la descripción (Huaraucaca, UM Colquijirca, Pasco)")
    if str(F.loc[e, "DEPARTAMENTO"]) == "NA":
        partes.append("departamento NA en el dato")
    nota[e] = "; ".join(partes)
F["zona_utm"] = [CORRECCION_ZONA.get(e, z) for e, z in zip(F.index, F["ZONA"])]
lat, lon = al.utm_a_latlon(pd.to_numeric(F["ESTE"]), pd.to_numeric(F["NORTE"]), pd.to_numeric(F["zona_utm"]), True)
F["lat"], F["lon"] = np.round(lat, 5), np.round(lon, 5)
if "CA-HUAR-01" in F.index:
    la19, lo19 = al.utm_a_latlon([float(F.loc["CA-HUAR-01", "ESTE"])], [float(F.loc["CA-HUAR-01", "NORTE"])], [19.0], True)
    print(f"CA-HUAR-01 con la zona 19 del dato caería en ({la19[0]:.3f}, {lo19[0]:.3f}); con zona 18 en "
          f"({F.loc['CA-HUAR-01', 'lat']:.3f}, {F.loc['CA-HUAR-01', 'lon']:.3f}).")
F["ubigeo"] = [u.zfill(6) if str(u).isdigit() else u for u in F["UBIGEO"].astype(str)]
F["nota_ubicacion"] = pd.Series(nota)

evs = L.groupby(["estacion", "evaluacion"], observed=True).size().rename("n").reset_index().sort_values(["estacion", "n"], ascending=[True, False])
F["evaluacion"] = evs.groupby("estacion", observed=True)["evaluacion"].agg(lambda s: " | ".join(map(str, s)))
F["objeto_fiscalizado"] = evs.groupby("estacion", observed=True)["evaluacion"].agg(
    lambda s: " | ".join(dict.fromkeys(objeto_fiscalizado(x) for x in s)))
F["tipo_actividad"] = evs.groupby("estacion", observed=True)["evaluacion"].agg(lambda s: tipo_actividad(" | ".join(map(str, s))))

g = L.groupby("estacion", observed=True)
F["fecha_inicio"] = g["dia"].min().dt.date
F["fecha_fin"] = g["dia"].max().dt.date
F["dias_con_filas"] = g["dia"].nunique()
V = L[L["estado"] == "valida"]
gv = V.groupby("estacion", observed=True)
F["dias_con_datos"] = gv["dia"].nunique()
F["anios_con_datos"] = gv["anio"].agg(lambda s: ", ".join(str(int(a)) for a in sorted(s.dropna().unique())))
F["parametros_medidos"] = gv["p"].agg(lambda s: ", ".join(sorted(s.unique())))
F["filas_largas"] = g.size()
F["filas_validas"] = gv.size()
F["pct_dias_en_cero"] = 100 * dia_p.groupby("estacion", observed=True)["cero"].mean()
F["pct_promedios_no_horarios"] = 100 * (~L["horaria"]).groupby(L["estacion"], observed=True).mean()
F["pct_filas_sin_valor"] = 100 * vacias.groupby(L["estacion"], observed=True).mean()
F["tiene_pm2_5"] = gv["p"].agg(lambda s: "pm2_5" in set(s))
F["tiene_pm10"] = gv["p"].agg(lambda s: "pm10" in set(s))
F["tiene_meteorologia"] = gv["p"].agg(lambda s: bool({"temperatura", "humedad", "viento_velocidad"} & set(s)))
F["promedios_24h_con_valor"] = L[(L["per"] == "24 horas") & (L["valor"] > 0)].groupby("estacion", observed=True).size()
if S is not None:
    gs = S.groupby("estacion")
    F["horas_validas"] = gs.size()
    F["pct_horas_inferidas"] = 100 * gs["hora_inferida"].mean()
else:  # sin silver de Go: horas distintas con alguna lectura válida (la misma cuenta, hecha aquí)
    F["horas_validas"] = V.groupby("estacion", observed=True)["fecha_hora"].nunique()
    F["pct_horas_inferidas"] = 100 * (1 - V.groupby("estacion", observed=True)["hora_explicita"].mean())
ruta_gold, ruta_asig, ruta_reg = DATA / "gold" / "oefa_features.csv", DATA / "gold" / "asignaciones.csv", REPORTS / "regimenes.json"
REG_ALTO = None
if ruta_gold.exists():
    F["filas_gold"] = pd.read_csv(ruta_gold, usecols=["estacion"]).groupby("estacion").size()
if ruta_asig.exists() and ruta_reg.exists():
    regs = json.loads(ruta_reg.read_text(encoding="utf-8"))["regimenes"]
    REG_ALTO = max(regs, key=lambda r: (r.get("pm10_medio") or 0, r.get("pm2_5_medio") or 0))["regimen"]
    A = pd.read_csv(ruta_asig)
    F["pct_regimen_alto"] = 100 * A.groupby("estacion")["regimen"].apply(lambda s: (s == REG_ALTO).mean())
    print(f"régimen alto = {REG_ALTO} (el de mayor PM10 medio en reports/regimenes.json)")
for c in ["horas_validas", "dias_con_datos", "filas_validas", "filas_gold", "promedios_24h_con_valor"]:
    if c in F:
        F[c] = F[c].fillna(0).astype(int)
for c in ["tiene_pm2_5", "tiene_pm10", "tiene_meteorologia"]:
    F[c] = F[c].astype("boolean").fillna(False).astype(bool)


def categoria(r):
    if r["horas_validas"] > 0 and (r["tiene_pm10"] or r["tiene_pm2_5"]):
        return "horarias con PM"
    if r["horas_validas"] > 0:
        return "horarias sin PM"
    if r["promedios_24h_con_valor"] > 0:
        return "solo promedios de 24 h"
    return "sin lecturas válidas"


F["categoria_mapa"] = F.apply(categoria, axis=1)
span = (pd.to_datetime(F["fecha_fin"]) - pd.to_datetime(F["fecha_inicio"])).dt.days + 1
F["tipo_serie"] = np.select(
    [F["categoria_mapa"] == "solo promedios de 24 h", span >= 365 * 1.5, span >= 180, span >= 0],
    ["muestreo manual de 24 h", "serie de años", "serie de meses", "campaña de semanas"], "—")
# grupos a menos de 15 km
dist = pd.DataFrame(al.haversine_km(F["lat"].to_numpy()[:, None], F["lon"].to_numpy()[:, None],
                                    F["lat"].to_numpy()[None, :], F["lon"].to_numpy()[None, :]), index=F.index, columns=F.index)
GRUPOS15 = al.grupos_cercanos(dist, 15.0)
F["grupo_15km"] = ""
for gr in GRUPOS15:
    F.loc[gr, "grupo_15km"] = " + ".join(gr) if len(gr) <= 3 else f"{gr[0]} y {len(gr) - 1} más"
F["formato_exportacion"] = FORMATO["formato"]
F = F.rename(columns={"DEPARTAMENTO": "departamento", "PROVINCIA": "provincia", "DISTRITO": "distrito",
                      "ALTITUD": "altitud", "DESCRIPCION_UBICACION": "descripcion_ubicacion"})
FICHA_COLS = ["evaluacion", "objeto_fiscalizado", "tipo_actividad", "departamento", "provincia", "distrito", "ubigeo",
              "lat", "lon", "altitud", "zona_utm", "nota_ubicacion", "fecha_inicio", "fecha_fin", "dias_con_filas",
              "dias_con_datos", "anios_con_datos", "horas_validas", "parametros_medidos", "filas_largas", "filas_validas",
              "pct_horas_inferidas", "pct_dias_en_cero", "pct_promedios_no_horarios", "pct_filas_sin_valor", "tiene_pm2_5",
              "tiene_pm10", "tiene_meteorologia", "promedios_24h_con_valor", "filas_gold", "pct_regimen_alto",
              "categoria_mapa", "tipo_serie", "grupo_15km", "formato_exportacion", "descripcion_ubicacion"]
FICHA = F[[c for c in FICHA_COLS if c in F]].copy()
FICHA["descripcion_ubicacion"] = FICHA["descripcion_ubicacion"].map(lambda s: " ".join(str(s).split()))
for c in ["pct_horas_inferidas", "pct_dias_en_cero", "pct_promedios_no_horarios", "pct_filas_sin_valor", "pct_regimen_alto"]:
    if c in FICHA:
        FICHA[c] = FICHA[c].round(2)
FICHA.sort_index().to_csv(REPORTS / "estaciones_ficha.csv", encoding="utf-8")
print(f"reports/estaciones_ficha.csv: {len(FICHA)} estaciones × {FICHA.shape[1]} columnas")
print(FICHA["categoria_mapa"].value_counts().to_dict())
print(FICHA["tipo_serie"].value_counts().to_dict())
tabla(FICHA[["departamento", "tipo_actividad", "fecha_inicio", "fecha_fin", "horas_validas", "filas_gold",
             "pct_regimen_alto", "categoria_mapa", "tipo_serie"]].sort_values("fecha_inicio").reset_index())

### 9.1 Faltantes por estación (tabla de la figura)

Cada fila larga cae en un destino: válida, día en cero (incluye la presión en 0 que Go cuenta en
`fuera_de_rango`), hora en cero (`cero_como_faltante`: un 0 de un contaminante en un día con datos), fila
sin valor (`sin_valor`), promedio no horario, duplicado por unidad, u otra regla.

In [ ]:
dest = L["estado"].astype(str).map({"valida": "válida", "bloque_en_cero": "día en cero", "cero_como_faltante": "hora en cero",
                                     "sin_valor": "fila sin valor", "periodo": "promedio no horario",
                                     "duplicado_unidad": "duplicado por unidad"}).fillna("otra regla")
dest[(L["estado"] == "fuera_de_rango") & (L["valor"] == 0)] = "día en cero"
dest[(L["estado"] == "periodo") & vacias] = "fila sin valor"
dest[(L["estado"] == "periodo") & L["horaria"] & ~vacias] = "otra regla"  # '-' en un día incompleto: hora, no periodo
DESTINOS = ["válida", "día en cero", "hora en cero", "fila sin valor", "promedio no horario", "duplicado por unidad", "otra regla"]
POR_EST = pd.crosstab(L["estacion"], dest).reindex(columns=DESTINOS, fill_value=0)
POR_EST["total"] = POR_EST.sum(axis=1)
POR_EST.to_csv(REPORTS / "faltantes_por_estacion.csv")
tot = POR_EST[DESTINOS].sum()
md("**Total del bronze por destino**: " + "; ".join(f"{k} {al.miles(v)} ({v / tot.sum():.1%})" for k, v in tot.items()))
otra = L.loc[dest == "otra regla", "estado"].astype(str).value_counts().to_dict()
print("«otra regla» se reparte en:", otra)

## 10. Figuras del informe

### 10.1 `faltantes.png`: a dónde va cada fila larga, y la regla de las 18 horas

In [ ]:
# faltantes.png y regla18.png: dos figuras al ancho final del informe (16 cm). Mismos colores que el
# calendario (fig. 3): válida azul, día en cero naranja, fila sin valor gris, promedios de 24 h verde,
# otra regla gris claro. Las notas van al caption del informe.
COL_DEST = dict(zip(DESTINOS, [al.PALETA[0], al.PALETA[1], al.PALETA[6], "#8c8b86", al.PALETA[2], al.PALETA[4], "#dcdbd5"]))
ORDEN = FICHA.assign(_i=pd.to_datetime(FICHA["fecha_inicio"])).sort_values(["_i"]).index.tolist()
pe_ = POR_EST.loc[ORDEN[::-1]]
frac_ = pe_[DESTINOS].div(pe_["total"], axis=0)
fig, ax = plt.subplots(figsize=(6.3, 0.128 * len(pe_) + 0.75))
izq = np.zeros(len(pe_))
for d_ in DESTINOS:
    ax.barh(range(len(pe_)), frac_[d_], left=izq, color=COL_DEST[d_], height=0.78, label=d_, edgecolor=al.SUPERFICIE, linewidth=0.4)
    izq += frac_[d_].to_numpy()
ax.set_yticks(range(len(pe_)))
ax.set_yticklabels([f"{e}  ({t / 1000:.0f} k)" if t >= 10_000 else f"{e}  ({al.miles(t)})" for e, t in zip(pe_.index, pe_["total"])], fontsize=6.3)
ax.set_xlim(0, 1)
ax.set_ylim(-0.6, len(pe_) - 0.4)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{100 * x:.0f} %"))
ax.tick_params(axis="x", labelsize=7)
ax.grid(axis="y", visible=False)
ax.set_xlabel("% de las filas largas de la estación (entre paréntesis, filas)", fontsize=7.5)
ax.legend(ncol=4, loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False, fontsize=6.8, handlelength=1.1, columnspacing=1.2, handletextpad=0.5)
fig.tight_layout()
guardar(fig, "faltantes.png", f"bronze {BRONZE.name} (todas las columnas), cascada de Go replicada fila por fila", dpi=300)

r18 = REGLA18
fig, bx = plt.subplots(figsize=(6.3, 2.4))
xx = np.arange(len(r18))
bx.bar(xx - 0.2, r18["pct_24h_en_cero"], width=0.4, color=al.PALETA[1], label="el promedio de 24 h publicado es 0")
bx.bar(xx + 0.2, r18["pct_24h_igual_media"], width=0.4, color=al.PALETA[0], label="el promedio publicado = media de las horas no nulas")
for i, n_ in enumerate(r18["dias"]):
    bx.text(i, 101, al.miles(n_), ha="center", va="bottom", fontsize=6, color=al.TINTA_2)
bx.axvline(3.5, color=al.TINTA_2, lw=0.9, ls="--")
bx.text(3.5, 111, "18 de 24 h (75 %)", fontsize=6.5, color=al.TINTA_2, va="bottom", ha="center", bbox=dict(facecolor=al.SUPERFICIE, edgecolor="none", pad=1))
bx.set_xticks(xx)
bx.set_xticklabels(r18["horas_no_nulas"].astype(str), fontsize=7)
bx.tick_params(axis="y", labelsize=7)
bx.set_ylim(0, 122)
bx.set_xlabel("horas del día con valor distinto de 0", fontsize=7.5)
bx.set_ylabel("% de los días (arriba, días comparados)", fontsize=7)
bx.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, fontsize=6.5, frameon=False, handlelength=1.2)
fig.tight_layout()
guardar(fig, "regla18.png", f"bronze {BRONZE.name}: días con 24 filas horarias y promedio de 24 h publicado", dpi=300)


### 10.2 `calendario.png`: disponibilidad por estación y día

Dos franjas por estación: **PM** (PM10 o PM2.5) y **meteorología** (temperatura, humedad, viento,
presión). Cada día toma el estado de más prioridad: lectura horaria válida > promedio de 24 h con valor
y sin horario (muestreo manual) > día en cero > fila sin valor (plantilla 2022) > descartado por otra regla
(día incompleto, fuera de rango) > sin filas.

In [ ]:
GRUPOS_CAL = {"PM": ["pm10", "pm2_5"], "Meteo": ["temperatura", "humedad", "viento_velocidad", "viento_direccion", "presion"]}
ESTADOS_CAL = ["sin filas", "otra regla", "fila sin valor", "día en cero", "solo 24 h (manual)", "lectura válida"]
COL_CAL = ["#fcfcfb", "#dcdbd5", "#8c8b86", al.PALETA[1], al.PALETA[2], al.PALETA[0]]
dias = pd.date_range("2018-01-01", "2022-12-31", freq="D")
prio = pd.Series(range(len(ESTADOS_CAL)), index=ESTADOS_CAL)
estado_fila = pd.Series("otra regla", index=L.index)
estado_fila[dest.isin(["día en cero", "hora en cero"])] = "día en cero"  # en el día, la hora en cero es «en cero»
estado_fila[dest == "fila sin valor"] = "fila sin valor"
estado_fila[(~L["horaria"]) & (L["valor"] > 0)] = "solo 24 h (manual)"
estado_fila[L["estado"] == "valida"] = "lectura válida"
estado_fila[(~L["horaria"]) & (L["valor"] > 0) & ~L["estacion"].isin(sin_h)] = "otra regla"  # 24 h junto a horarios
codigo = estado_fila.map(prio)
filas_cal, etiquetas = [], []
M = np.zeros((len(ORDEN) * 2, len(dias)), dtype=int)
for i, e in enumerate(ORDEN):
    for j, (gname, params) in enumerate(GRUPOS_CAL.items()):
        m = (L["estacion"] == e) & L["p_inf"].isin(params) & L["dia"].notna()
        s = codigo[m].groupby(L.loc[m, "dia"]).max()
        M[2 * i + j] = s.reindex(dias).fillna(0).astype(int).to_numpy()
from matplotlib.colors import ListedColormap
fig, ax = plt.subplots(figsize=(13, 12.5))
ax.imshow(M, aspect="auto", interpolation="nearest", cmap=ListedColormap(COL_CAL), vmin=0, vmax=len(COL_CAL) - 1,
          extent=(0, len(dias), len(ORDEN) * 2, 0))
for i in range(1, len(ORDEN)):
    ax.axhline(2 * i, color=al.TINTA_2, lw=0.6)
for i in range(len(ORDEN)):
    ax.axhline(2 * i + 1, color="#b5b4ae", lw=0.4, ls=":")
x22 = (pd.Timestamp("2022-01-01") - dias[0]).days
ax.axvline(x22, color=al.TINTA, lw=1.0, ls="--")
ax.annotate("desde 2022 lo que falta llega como fila vacía (gris);\nhasta 2021, como día en 0 (naranja) →", (x22, 2 * len(ORDEN) - 2),
            xytext=(-10, 0), textcoords="offset points", ha="right", va="center", fontsize=8.5, color=al.TINTA,
            path_effects=[pe.withStroke(linewidth=3, foreground=al.SUPERFICIE)])
ax.set_yticks([2 * i + 1 for i in range(len(ORDEN))])
ax.set_yticklabels([f"{e} · {al.NOMBRE_REGION.get(al.region(FICHA.loc[e, 'departamento']) or '', 'NA')}" for e in ORDEN], fontsize=8)
for i in range(len(ORDEN)):
    ax.text(len(dias) + 8, 2 * i + 0.5, "PM", fontsize=6, va="center", color=al.TINTA_2)
    ax.text(len(dias) + 8, 2 * i + 1.5, "met", fontsize=6, va="center", color=al.TINTA_2)
anios = pd.date_range("2018-01-01", "2023-01-01", freq="YS")
ax.set_xticks([(a - dias[0]).days for a in anios])
ax.set_xticklabels([str(a.year) for a in anios])
ax.set_xticks([(a - dias[0]).days for a in pd.date_range("2018-01-01", "2022-12-31", freq="QS")], minor=True)
ax.grid(axis="x", which="major", color=al.TINTA_2, lw=0.6, alpha=0.5)
ax.grid(axis="y", visible=False)
ax.set_xlim(0, len(dias))
ax.set_title("Disponibilidad por estación y día, 2018–2022 (dos franjas por estación: PM y meteorología)", loc="left")
ax.legend(handles=[mpatches.Patch(facecolor=c, edgecolor="#b5b4ae", label=k) for k, c in zip(ESTADOS_CAL[::-1], COL_CAL[::-1])],
          ncol=6, loc="lower left", bbox_to_anchor=(0, 1.02), frameon=False, fontsize=8.5)
fig.tight_layout()
al.nota_al_pie(fig, "Fuente: OEFA, Vigilancia y Seguimiento ambiental en la calidad del aire (corte 2024-04-29). Estaciones ordenadas por su primer día. «lectura válida»: alguna lectura horaria que pasa "
               "la cascada de Go; «solo 24 h»: promedio de 24 h > 0\nsin serie horaria (muestreo manual de Ventanilla-Mi Perú y CA-01); «fila sin valor»: hueco de la plantilla de exportación de 2022; "
               "«otra regla»: día incompleto o intercalado hora a hora (CA-CH-03, ago–oct 2020), fuera de rango.")
guardar(fig, "calendario.png", f"bronze {BRONZE.name} (todas las columnas), cascada de Go replicada fila por fila")

### 10.3 Estructura temporal: campañas de semanas, series de meses y series de años

In [ ]:
est_tipo = FICHA.groupby("tipo_serie").agg(estaciones=("horas_validas", "size"), horas_validas=("horas_validas", "sum"),
                                           filas_gold=("filas_gold", "sum") if "filas_gold" in FICHA else ("horas_validas", "size"))
est_tipo["% del gold"] = 100 * est_tipo["filas_gold"] / max(est_tipo["filas_gold"].sum(), 1)
tabla(est_tipo.reset_index())
for t_ in ["campaña de semanas", "serie de meses", "serie de años", "muestreo manual de 24 h"]:
    print(f"{t_}: {', '.join(FICHA.index[FICHA['tipo_serie'] == t_])}")
dic21 = FICHA[(pd.to_datetime(FICHA["fecha_inicio"]) >= "2021-11-01") & (pd.to_datetime(FICHA["fecha_inicio"]) < "2022-01-01")]
print(f"estaciones que empiezan en nov-dic 2021: {len(dic21)} ({', '.join(dic21.index)})")
if ruta_asig.exists():
    print("filas del gold por año:", A.assign(anio=A["fecha_hora"].str[:4]).groupby("anio").size().to_dict())

### 10.4 Mapa interactivo: `reports/mapa_estaciones.html`

Leaflet (folium) sobre OpenStreetMap. Color por categoría, tamaño por horas válidas (raíz cuadrada).
Las estaciones a menos de 15 km entre sí se agrupan en un racimo que se abre al acercarse o al hacer clic.
Al pasar el mouse: nombre y categoría; al hacer clic: la ficha.

In [ ]:
import folium
from folium.plugins import MarkerCluster

CATEGORIAS = ["horarias con PM", "horarias sin PM", "solo promedios de 24 h", "sin lecturas válidas"]
COL_CAT = dict(zip(CATEGORIAS, [al.PALETA[0], al.PALETA[1], al.PALETA[2], "#8c8b86"]))
hmax = max(FICHA["horas_validas"].max(), 1)


def radio_px(h):
    return 9 + 22 * np.sqrt(h / hmax)


def txt(v, vacio="—"):
    return vacio if v is None or (isinstance(v, float) and np.isnan(v)) or v == "" else str(v)


def ficha_html(e, r):
    def fila(k, v):
        return f"<tr><th style='text-align:left;padding-right:8px;vertical-align:top;color:#52514e'>{k}</th><td>{v}</td></tr>"
    ev_ = html.escape(str(r["evaluacion"])).replace(" | ", "<br>")
    alto = (f"{r['pct_regimen_alto']:.1f} % de {al.miles(r['filas_gold'])} h del gold" if "pct_regimen_alto" in r and pd.notna(r.get("pct_regimen_alto"))
            else "no está en el gold (le falta PM2.5 o PM10 horario)")
    ubic_ = f"{r['departamento']} / {r['provincia']} / {r['distrito']}"
    if txt(r.get("nota_ubicacion"), ""):
        ubic_ += f"<br><span style='color:#a32626'>{html.escape(r['nota_ubicacion'])}</span>"
    t = (f"<div style='font-family:system-ui,sans-serif;font-size:12px;max-width:380px'>"
         f"<div style='font-size:15px;font-weight:700;margin-bottom:4px'>{e}</div>"
         f"<div style='color:{COL_CAT[r['categoria_mapa']]};font-weight:600;margin-bottom:6px'>{r['categoria_mapa']} · {r['tipo_serie']}</div>"
         "<table style='border-collapse:collapse'>"
         + fila("Evaluación", ev_)
         + fila("Actividad", html.escape(f"{r['objeto_fiscalizado']} ({r['tipo_actividad']})"))
         + fila("Dpto. / prov. / distrito", ubic_)
         + fila("Altitud", f"{r['altitud']} m s. n. m.")
         + fila("Datos", f"{r['fecha_inicio']} a {r['fecha_fin']}; años con datos válidos: {txt(r['anios_con_datos'])}")
         + fila("Horas válidas", al.miles(r["horas_validas"]) + (f" ({r['pct_horas_inferidas']:.0f} % con hora inferida)" if r["horas_validas"] else
                                                                  f" (promedios de 24 h con valor: {r['promedios_24h_con_valor']})"))
         + fila("Parámetros", html.escape(txt(r["parametros_medidos"])))
         + fila("Régimen alto", alto)
         + fila("Días en cero", f"{r['pct_dias_en_cero']:.1f} % de los días-parámetro" if pd.notna(r["pct_dias_en_cero"]) else "—")
         + fila("Ubicación", html.escape(r["descripcion_ubicacion"]))
         + "</table></div>")
    return t


mapa = folium.Map(location=[-10.2, -75.8], zoom_start=6, tiles="OpenStreetMap", control_scale=True)
icono_racimo = """function(cluster) {
  var n = cluster.getChildCount();
  return L.divIcon({html: '<div style="background:#0b0b0b;color:#fcfcfb;border:2px solid #fcfcfb;border-radius:50%;'
    + 'width:34px;height:34px;line-height:30px;text-align:center;font:600 13px system-ui;box-shadow:0 0 0 1px #52514e">' + n + '</div>',
    className: '', iconSize: L.point(34, 34)});
}"""
capas = {}
for gr in GRUPOS15:
    capas[tuple(gr)] = MarkerCluster(name=" + ".join(gr) if len(gr) <= 3 else f"{gr[0]} y {len(gr) - 1} más",
                                     icon_create_function=icono_racimo,
                                     options={"maxClusterRadius": 120, "spiderfyOnMaxZoom": True, "showCoverageOnHover": False,
                                              "disableClusteringAtZoom": 16}).add_to(mapa)
for e, r in FICHA.iterrows():
    d_ = 2 * radio_px(r["horas_validas"])
    icono = folium.DivIcon(html=f"<div style='width:{d_:.0f}px;height:{d_:.0f}px;border-radius:50%;background:{COL_CAT[r['categoria_mapa']]};"
                                f"opacity:0.85;border:2px solid #fcfcfb;box-shadow:0 0 0 1px #52514e'></div>",
                           icon_size=(d_, d_), icon_anchor=(d_ / 2, d_ / 2))
    destino = next((capas[tuple(gr)] for gr in GRUPOS15 if e in gr), mapa)
    folium.Marker([r["lat"], r["lon"]], icon=icono,
                  tooltip=f"<b>{e}</b> · {r['categoria_mapa']} · {al.miles(r['horas_validas'])} h válidas",
                  popup=folium.Popup(ficha_html(e, r), max_width=420)).add_to(destino)
leyenda = ("<details open style='position:fixed;bottom:24px;left:12px;z-index:9999;background:#fcfcfb;padding:8px 12px;border:1px solid #b5b4ae;"
           "border-radius:6px;font:12px system-ui,sans-serif;color:#0b0b0b;max-width:290px'>"
           "<summary style='font-weight:700;cursor:pointer;margin-bottom:4px'>Estaciones de OEFA (2018–2022)</summary>"
           + "".join(f"<div><span style='display:inline-block;width:11px;height:11px;border-radius:50%;background:{COL_CAT[c]};margin-right:6px'></span>"
                     f"{c} ({int((FICHA['categoria_mapa'] == c).sum())})</div>" for c in CATEGORIAS)
           + "<div style='margin-top:4px;color:#52514e'>Tamaño: horas válidas. Número negro: racimo de estaciones a menos de 15 km (clic para abrir).</div>"
           "<div style='margin-top:4px;color:#a32626'>Una zona sin estación no es una zona con aire limpio: las estaciones están donde OEFA "
           "fiscaliza una actividad, no donde vive la gente.</div></details>")
mapa.get_root().html.add_child(folium.Element(leyenda))
mapa.get_root().header.add_child(folium.Element("<title>Estaciones de calidad del aire de OEFA</title>"))
mapa.save(str(REPORTS / "mapa_estaciones.html"))
print("reports/mapa_estaciones.html:", f"{(REPORTS / 'mapa_estaciones.html').stat().st_size / 1e3:.0f} kB")
mapa

### 10.5 `mapa.png`: mapa estático para el informe

Teselas de OpenTopoMap (datos de OpenStreetMap y relieve) con `contextily`: dan límites, nombres y la
cordillera, que importa porque 10 de las 33 estaciones están sobre los 3 000 m.
Los servidores de tile.openstreetmap.org rechazan la descarga por script (política de uso), por eso no se
usan aquí; el mapa interactivo sí usa OpenStreetMap, que se carga en el navegador. Sin `contextily` o sin
red, el mapa sale sin fondo, con las etiquetas. Dos recuadros: Ventanilla–Mi Perú y Huachipa–Cajamarquilla.

In [ ]:
try:
    import contextily as cx
except Exception:
    cx = None
R_T = 6378137.0


def merc(lat, lon):
    lat, lon = np.asarray(lat, float), np.asarray(lon, float)
    return R_T * np.radians(lon), R_T * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))


FICHA["x"], FICHA["y"] = merc(FICHA["lat"], FICHA["lon"])


FONDO = cx.providers.OpenTopoMap if cx is not None else None  # relieve y nombres de lugar, aclarado para que manden los puntos


def fondo(ax, zoom):
    if cx is None:
        return "sin contextily: mapa sin fondo"
    try:
        # OpenTopoMap (datos de OpenStreetMap y relieve SRTM). Los servidores de tile.openstreetmap.org
        # rechazan (403) la descarga por script y CartoDB ya exige clave.
        cx.add_basemap(ax, crs="EPSG:3857", source=FONDO, zoom=zoom, attribution=False, alpha=0.6)
        return "teselas OpenTopoMap"
    except Exception as ex:  # sin red
        return f"sin teselas ({type(ex).__name__})"


def radio_px_mapa(h):
    return 4 + 9 * np.sqrt(h / hmax)  # radio en puntos, para una figura de 15 cm


def puntos(ax, df, escala=1.0):
    for c in CATEGORIAS:
        s = df[df["categoria_mapa"] == c]
        if len(s):
            ax.scatter(s["x"], s["y"], s=(escala * radio_px_mapa(s["horas_validas"])) ** 2, c=COL_CAT[c], alpha=0.9,
                       edgecolors="white", linewidths=0.6, zorder=5, label=f"{c} ({len(s)})")


def rotulo(ax, x, y, texto, dx=0, dy=0, ha="left", tam=6.5):
    ax.annotate(texto, (x, y), xytext=(dx, dy), textcoords="offset points", ha=ha, va="center", fontsize=tam, color=al.TINTA, zorder=6,
                path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])


def nombre_grupo(gr):
    if len(gr) <= 3:
        return " + ".join(gr)
    distritos = sorted(set(FICHA.loc[gr, "distrito"].astype(str).str.title()))
    return f"{' / '.join(distritos)} ({len(gr)})"


SUELTAS = [[e] for e in FICHA.index if not any(e in g_ for g_ in GRUPOS15)]
# Dibujada al ancho final del informe (15 cm): el Perú entero (la Amazonía sin estaciones es parte del
# mensaje), sin Ecuador, sin título ni texto dentro de la imagen (van al caption).
fig = plt.figure(figsize=(5.9, 4.7))
ax = fig.add_axes([0.0, 0.0, 0.575, 1.0])
x0, y0 = merc(-18.55, -81.6)
x1, y1 = merc(-3.1, -68.5)
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)
ax.set_aspect("equal", adjustable="box")
estado_fondo = fondo(ax, 7)
puntos(ax, FICHA)
ROT = {"CA-TA-01": (6, 3), "CA-HUA-01": (6, -5), "CA-CG-01": (6, 0), "CA-01": (6, 0), "CA-AYAS-01": (6, -2),
       "CA-COLQ-01": (6, 4), "CA-CC-01": (6, -2), "CA-VMP-1": (-6, 1), "CA-HU-04": (6, -5), "CA-CP-01": (6, 4),
       "CA-VE-01": (-6, -3), "CA-ESP-01": (6, -3), "CA-SOC-01": (6, 0), "CA-TO-01": (-6, 0), "CA-TAL-01": (6, -3)}
for gr in GRUPOS15 + SUELTAS:
    gx, gy = FICHA.loc[gr, "x"].mean(), FICHA.loc[gr, "y"].mean()
    dx, dy = ROT.get(gr[0], (6, 0))
    rotulo(ax, gx, gy, nombre_grupo(gr), dx=dx, dy=dy, ha="right" if dx < 0 else "left")
ax.set_xticks([])
ax.set_yticks([])
for s_ in ax.spines.values():
    s_.set_visible(True)
    s_.set_color(al.REJILLA)
ax.legend(loc="lower left", fontsize=6, frameon=True, facecolor="white", framealpha=0.9, title="color: categoría · tamaño: horas válidas",
          title_fontsize=6, markerscale=0.7, borderpad=0.6, handletextpad=0.4)


def recuadro(pos, estaciones, zoom, titulo, rot):
    bx = fig.add_axes(pos)
    sub = FICHA.loc[estaciones]
    padx = max(sub["x"].max() - sub["x"].min(), 600) * 0.28 + 250
    pady = max(sub["y"].max() - sub["y"].min(), 600) * 0.28 + 250
    bx.set_xlim(sub["x"].min() - padx, sub["x"].max() + padx)
    bx.set_ylim(sub["y"].min() - pady, sub["y"].max() + pady)
    bx.set_aspect("equal", adjustable="box")
    fondo(bx, zoom)
    puntos(bx, sub, escala=0.8)
    for e, r in sub.iterrows():
        dx, dy = rot.get(e, (6, 0))
        rotulo(bx, r["x"], r["y"], e, dx=dx, dy=dy, ha="right" if dx < 0 else "left", tam=6)
    bx.set_xticks([])
    bx.set_yticks([])
    for s_ in bx.spines.values():
        s_.set_visible(True)
        s_.set_color(al.TINTA_2)
    bx.set_title(titulo, loc="left", fontsize=7, pad=3)
    return bx


VMP = [e for e in FICHA.index if e.startswith("CA-VMP")]
HUA = [e for e in FICHA.index if e.startswith(("CA-HU-", "CA-RZC"))]
recuadro([0.615, 0.50, 0.385, 0.455], VMP, 16, "Ventanilla–Mi Perú: 8 puntos en 1,5 km",
         {"CA-VMP-1": (6, 0), "CA-VMP-10": (-6, -5), "CA-VMP-2": (-6, 5), "CA-VMP-2A": (6, 5), "CA-VMP-6": (6, 0),
          "CA-VMP-7": (6, -5), "CA-VMP-8": (-6, 4), "CA-VMP-9": (-6, -4)})
recuadro([0.615, 0.0, 0.385, 0.455], HUA, 14, "Huachipa–Cajamarquilla (Lima)",
         {"CA-HU-04": (6, 0), "CA-HU-09": (-6, 0), "CA-RZC-01": (6, 0), "CA-RZC-02": (6, 0)})
cuenta = FICHA.groupby("departamento").size().sort_values(ascending=False)
print("Estaciones por departamento (para el caption):", ", ".join(f"{al.NOMBRE_REGION.get(al.region(d_) or '', d_)} {n_}" for d_, n_ in cuenta.items()))
guardar(fig, "mapa.png", "bronze (ubicación por estación) y silver/gold de Go (horas válidas)", dpi=300)
print("fondo:", estado_fondo)


## 11. Unidad de análisis en cada capa

In [ ]:
capas_t = [{"capa": "bronze (fila larga)", "unidad": "una lectura de un parámetro en una estación, un día y una posición (hora) — o un promedio de 24 h/móvil",
            "filas": len(L)},
           {"capa": "bronze válido (Go)", "unidad": "lectura horaria que pasa la cascada", "filas": int((L["estado"] == "valida").sum())}]
if S is not None:
    capas_t.append({"capa": "silver (fila ancha)", "unidad": "estación × hora, con los parámetros que haya (columnas vacías si no)", "filas": len(S)})
if ruta_gold.exists():
    capas_t.append({"capa": "gold", "unidad": "estación × hora con PM2.5 y PM10 válidos, estandarizada (lo que ve el K-Means)",
                    "filas": int(FICHA["filas_gold"].sum())})
tabla(pd.DataFrame(capas_t))
print(f"estaciones: bronze {L['estacion'].nunique()}, con lecturas válidas {int((FICHA['horas_validas'] > 0).sum())}, "
      f"en el gold {int((FICHA.get('filas_gold', pd.Series(0)) > 0).sum())}")
print("archivos:", ", ".join(sorted(p.name for p in REPORTS.glob("faltantes_*.csv"))), "| estaciones_ficha.csv | mapa_estaciones.html")

La lectura de todo esto (qué es una EAS, por qué faltan valores, qué implica para el modelo y para la
pregunta de impacto, con las fuentes de OEFA y MINAM citadas) está en `docs/eda-procedencia.md`.